# BAIXANDO O DATASET

Este código baixa somente 10 sujeitos do datatset

In [2]:
!pip install boto3 botocore

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
import random



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/140.6 kB ? eta -:--:--
   -- ------------------------------------- 10.2/140.6 kB ? eta -:--:--
   ------------------- ------------------- 71.7/140.6 kB 975.2 kB/s eta 0:00:01
   ---------------------------------------- 140.6/140.6 kB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   - -------------------------------------- 0.4/14.9 MB 8.1 MB/s eta 0:00:02
   ----- ---------------------------------- 2.0/14.9 MB 25.2 MB/s eta 0:00:01
   --------- ------------------------------ 3.5/14.9 MB 27.8 MB/s eta 0:00:01
   -------------- ------------------------- 5.5/14.9 MB 31.9 MB/s eta 0:00:01
   --------------------- ------------------ 8.1/14.9 MB 34.5 MB/s eta 0:00:01
   -------------------------- ------------- 9.8/14.9 MB 34.9 MB/s eta 0:00:01
   -------------------------------- ------- 12.2/14.9 MB 46.7 MB/s eta 0:00:01
   ------------------------------------ --- 13.7/14.9 MB 43.7 MB/s eta 0:00:01
  

In [3]:
# CONFIGURAÇÕES

bucket = "physionet-open"
base_prefix = "eegmmidb/1.0.0/"
local_root = "data_eegmmidb"

# Quantidade de pacientes que deseja baixar
NUM_PATIENTS = 10

# Seed fixa para garantir reprodutibilidade
# (todos que usarem essa mesma seed e mesmo NUM_PATIENTS
# terão exatamente os mesmos pacientes sorteados)
SEED = 42

# Total de sujeitos disponíveis no dataset: S001 até S109
TOTAL_PATIENTS = 109

# PREPARAÇÃO

os.makedirs(local_root, exist_ok=True)

s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED)
)

# SORTEIO REPRODUTÍVEL DOS PACIENTES

all_patients = [f"S{i:03d}" for i in range(1, TOTAL_PATIENTS + 1)]

random.seed(SEED)
selected_patients = sorted(random.sample(all_patients, NUM_PATIENTS))

print("Pacientes selecionados:")
print(selected_patients)

# DOWNLOAD DOS PACIENTES SELECIONADOS

for patient in selected_patients:
    prefix = f"{base_prefix}{patient}/"
    local_dir = os.path.join(local_root, patient)
    os.makedirs(local_dir, exist_ok=True)

    print(f"\n=== Verificando {patient} ===")

    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]

            if key.endswith("/"):
                continue

            local_path = os.path.join(
                local_dir,
                os.path.basename(key)
            )

            if os.path.exists(local_path):
                print("Skipping (já existe):", local_path)
                continue

            print("Downloading", key)
            s3.download_file(bucket, key, local_path)

print("\nDownload concluído.")

Pacientes selecionados:
['S004', 'S014', 'S015', 'S018', 'S029', 'S032', 'S036', 'S082', 'S087', 'S095']

=== Verificando S004 ===

=== Verificando S014 ===

=== Verificando S015 ===

=== Verificando S018 ===

=== Verificando S029 ===

=== Verificando S032 ===

=== Verificando S036 ===

=== Verificando S082 ===

=== Verificando S087 ===

=== Verificando S095 ===

Download concluído.
